


---
## Setup

Run the cell below to import required libraries and configure the environment.

In [21]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score
import shap
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('bank-additional-full.csv', sep=';')
df['y'] = df['y'].map({'yes': 1, 'no': 0}).astype(int)
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])
print(f"Loaded {len(df):,} records")

Loaded 41,188 records


## 1. Data Loading

Loads the bank marketing dataset. Automatically handles delimiter detection.

In [22]:

column = 'age'  # <-- CHANGE THIS

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if df[column].dtype in ['int64', 'float64']:
    df[column].hist(ax=axes[0], bins=30, edgecolor='black')
    axes[0].set_title(f'Distribution: {column}')
    df.boxplot(column=column, by='y', ax=axes[1])
    axes[1].set_title(f'{column} by Subscription')
else:
    df[column].value_counts().plot(kind='bar', ax=axes[0], edgecolor='black')
    axes[0].set_title(f'Count: {column}')
    axes[0].tick_params(axis='x', rotation=45)
    
    conv_rate = df.groupby(column)['y'].mean().sort_values()
    conv_rate.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
    axes[1].axhline(y=df['y'].mean(), color='red', linestyle='--')
    axes[1].set_title(f'Conversion Rate by {column}')

plt.tight_layout()
plt.show()

## 2. Data Exploration

Select a column to examine its distribution and relationship with the target.

In [23]:
def explore_column():
    """Display distribution and conversion rate for a selected column."""
    # === USER CONFIGURATION ===
    # Change this variable to analyze different columns
    selected_column = 'age'  # Options: age, job, marital, education, campaign, euribor3m
    # =========================
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    if df[selected_column].dtype in ['int64', 'float64']:
        # Numeric column
        df[selected_column].hist(ax=axes[0], bins=30, edgecolor='black')
        axes[0].set_title(f'Distribution: {selected_column}')
        
        df.boxplot(column=selected_column, by='y', ax=axes[1])
        axes[1].set_title(f'{selected_column} by Subscription')
        axes[1].set_xlabel('Subscribed (0=No, 1=Yes)')
    else:
        # Categorical column
        df[selected_column].value_counts().plot(kind='bar', ax=axes[0], edgecolor='black')
        axes[0].set_title(f'Count: {selected_column}')
        axes[0].tick_params(axis='x', rotation=45)
        
        conv_rate = df.groupby(selected_column)['y'].mean().sort_values()
        conv_rate.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
        axes[1].axhline(y=df['y'].mean(), color='red', linestyle='--', label=f'Avg: {df["y"].mean():.1%}')
        axes[1].set_title(f'Conversion Rate by {selected_column}')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].legend()
    
    plt.tight_layout()
    plt.show()

# Available columns for analysis
print("Available columns for analysis:")
print("- age (numeric)")
print("- job (categorical)")
print("- marital (categorical)") 
print("- education (categorical)")
print("- campaign (numeric)")
print("- euribor3m (numeric)")

# Run the exploration
explore_column()

Available columns for analysis:
- age (numeric)
- job (categorical)
- marital (categorical)
- education (categorical)
- campaign (numeric)
- euribor3m (numeric)


## 3. Customer Persona Discovery

Applies K-means clustering to identify customer segments based on demographics.

**Parameter**: `n_clusters` - Number of customer segments to identify.

In [24]:

n_clusters = 4  # <-- CHANGE THIS (2-8)

# Prepare data
demo_df = df[['age', 'job', 'marital', 'education']].copy()
for col in demo_df.select_dtypes('object'):
    demo_df[col] = LabelEncoder().fit_transform(demo_df[col].astype(str))

# Scale and cluster
X_scaled = StandardScaler().fit_transform(demo_df.fillna(demo_df.mean()))
km = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
df['Persona'] = km.fit_predict(X_scaled)

# Show results
conv_rates = df.groupby('Persona')['y'].mean().sort_values()
plt.figure(figsize=(8, 4))
plt.bar(range(len(conv_rates)), conv_rates.values, color='steelblue', edgecolor='black')
plt.axhline(y=df['y'].mean(), color='red', linestyle='--')
plt.xticks(range(len(conv_rates)), [f'P{i}' for i in conv_rates.index])
plt.ylabel('Conversion Rate')
plt.title(f'Conversion Rate by Persona (K={n_clusters})')
plt.show()

## 4. Scenario Analysis with SHAP

Analyze feature importance under different market conditions.

**Available Scenarios**:
- Global Baseline: Full dataset
- High Interest Rates: euribor3m > 3.0
- Low Interest Rates: euribor3m < 1.5
- Retirees: job == 'retired'
- Students: job == 'student'
- Young Adults: age < 35
- Seniors: age > 55
- First Contact: campaign == 1
- High Contact Fatigue: campaign > 3

In [26]:


# === CHOOSE SCENARIO (uncomment one) ===
# filter_expr = None                          # Global Baseline
# filter_expr = "euribor3m > 3.0"             # High Interest Rates
# filter_expr = "euribor3m < 1.5"             # Low Interest Rates
# filter_expr = "job == 'retired'"            # Retirees
# filter_expr = "job == 'student'"            # Students
# filter_expr = "age < 35"                    # Young Adults
# filter_expr = "age > 55"                    # Seniors
# filter_expr = "campaign == 1"               # First Contact
filter_expr = "campaign > 3"                  # High Contact Fatigue
# =====================================

# Filter data
if filter_expr:
    work_df = df.query(filter_expr).copy()
else:
    work_df = df.copy()

print(f"Samples: {len(work_df):,} | Conversion: {work_df['y'].mean():.1%}")

# Prepare features - drop Persona if it exists
drop_cols = ['y']
if 'Persona' in work_df.columns:
    drop_cols.append('Persona')

X = pd.get_dummies(work_df.drop(drop_cols, axis=1), drop_first=True)
y = work_df['y']

# Train model
model = RandomForestClassifier(n_estimators=100, max_depth=7, random_state=42)
model.fit(X, y)
print(f"AUC: {roc_auc_score(y, model.predict_proba(X)[:, 1]):.4f}")

# SHAP - use smaller sample for speed
X_sample = X.sample(min(200, len(X)), random_state=42)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

# For binary classification, shap_values is a list of two arrays
# Take class 1 values
if isinstance(shap_values, list):
    shap_vals = shap_values[1]
else:
    shap_vals = shap_values

# Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_sample, plot_type="bar", show=False)
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_sample, show=False)
plt.title("SHAP Summary")
plt.tight_layout()
plt.show()

Samples: 7,635 | Conversion: 7.3%
AUC: 0.8055


## 5. Fairness Audit (OSFI E-23)

Evaluates model bias across demographic groups. Required for regulatory compliance.

In [27]:
# Cell 5: Fairness Audit

# Train model on all data
X = pd.get_dummies(df.drop(['y', 'Persona'], axis=1), drop_first=True)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, df['y'])
df['Score'] = model.predict_proba(X)[:, 1]

# Age groups
df['AgeGroup'] = pd.cut(df['age'], bins=[0, 30, 45, 60, 100], 
                        labels=['18-30', '31-45', '46-60', '60+'])

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

age_scores = df.groupby('AgeGroup')['Score'].mean()
age_scores.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].axhline(y=df['Score'].mean(), color='red', linestyle='--')
axes[0].set_title('Mean Score by Age Group')
axes[0].set_ylabel('Predicted Probability')

marital_scores = df.groupby('marital')['Score'].mean().sort_values()
marital_scores.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].axhline(y=df['Score'].mean(), color='red', linestyle='--')
axes[1].set_title('Mean Score by Marital Status')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"Age disparity: {age_scores.max() - age_scores.min():.2%}")
print(f"Marital disparity: {marital_scores.max() - marital_scores.min():.2%}")

Age disparity: 36.43%
Marital disparity: 5.82%


## 6. Business Rules Engine

Converts model scores into actionable strategies with ROI calculation.

**Parameters**:
- Call Cost: Cost per outbound call
- Sale Value: Revenue from successful subscription

In [28]:

call_cost = 5    # <-- CHANGE THIS
sale_value = 300 # <-- CHANGE THIS

# Get scores from model
X = pd.get_dummies(df.drop(['y', 'Persona', 'Score'], axis=1, errors='ignore'), drop_first=True)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, df['y'])
df['Score'] = model.predict_proba(X)[:, 1]

# Simple rules
def get_action(row):
    if row['Score'] >= 0.80:
        return 'CALL_NOW'
    elif row['Score'] >= 0.55 and row['campaign'] <= 2:
        return 'NURTURE'
    elif row['Score'] >= 0.55:
        return 'EMAIL_ONLY'
    elif row['Score'] >= 0.30 and row['campaign'] <= 1:
        return 'ONE_MORE_TRY'
    else:
        return 'DO_NOT_CONTACT'

df['Action'] = df.apply(get_action, axis=1)

# Calculate profit
stats = df.groupby('Action').agg(
    Count=('y', 'count'),
    Conversions=('y', 'sum')
)
stats['Profit'] = stats['Conversions'] * sale_value - stats['Count'] * call_cost

print(f"Call Cost: ${call_cost} | Sale Value: ${sale_value}")
print(stats)

# Plot
plt.figure(figsize=(8, 4))
colors = ['green' if p > 0 else 'red' for p in stats['Profit']]
stats['Profit'].plot(kind='bar', color=colors, edgecolor='black')
plt.axhline(0, color='black', linewidth=0.5)
plt.title('Profit by Action')
plt.tight_layout()
plt.show()

Call Cost: $5 | Sale Value: $300
                Count  Conversions  Profit
Action                                    
CALL_NOW         1654         1654  487930
DO_NOT_CONTACT  36372           84 -156660
EMAIL_ONLY        810          809  238650
NURTURE          1941         1925  567795
ONE_MORE_TRY      411          168   48345


## 7. Run Complete Analysis

Executes all analyses sequentially and displays results.